# Pasado `gpu-hybrid`: prag za automatski izbor `gelsd` / eager A2 (v3 kod)

Pre pokretanja: **Runtime > Change runtime type > GPU (A100)**, zatim **Runtime > Disconnect and delete runtime**, pa **Runtime > Run all**.

Meri samo G (`gelsd` na CPU-u, sa prenosima) i eager A2 (`planar_fit.fit_planes`), bez Inductor-a i A3, za `n = 64 ... 2048` boksova po pozivu, `float64` i `float32`, u 3 kruga (5 warm-up, 20 merenih poziva). Pravilo za prag je zapisano i commitovano pre merenja, u zaglavlju `experiments/measure_plane_fit_crossover.py` (commit `8837f68`):

> najmanji `n` na kome je eager A2 bar 1.2x brži od G i nije sporiji od G ni na jednom većem testiranom `n`; stabilan ako se 3 kruga slažu najviše za jedan korak liste, i tada važi najveći od njih.

Trajanje: nekoliko minuta (procena). Kad završi, pošalji `plane_fit_crossover.zip`.

In [ ]:
import subprocess
import torch

subprocess.run(["nvidia-smi"], check=True)
assert torch.cuda.is_available(), "Uključi GPU: Runtime > Change runtime type > GPU"
GPU = torch.cuda.get_device_name(0)
print("Torch:", torch.__version__, "| CUDA:", torch.version.cuda, "| GPU:", GPU)
assert "A100" in GPU, f"Prag se meri na A100, a ovo je {GPU}."

In [ ]:
import os
import shutil

REPO = "/content/Pasado"
URL = "https://github.com/dimitrijepesic/Pasado.git"
COMMIT = "8837f68"

os.chdir("/content")
shutil.rmtree(REPO, ignore_errors=True)
subprocess.run(["git", "-c", "http.version=HTTP/1.1", "clone", "--branch", "gpu-hybrid",
                "--single-branch", URL, REPO], check=True)
subprocess.run(["git", "-C", REPO, "checkout", "-q", COMMIT], check=True)
head = subprocess.check_output(["git", "-C", REPO, "rev-parse", "HEAD"], text=True).strip()
assert head.startswith(COMMIT), f"HEAD je {head}, a ocekivan je {COMMIT}"
print("HEAD:", head)
os.chdir(REPO)
OUT = "/content/plane_fit_crossover"
os.makedirs(OUT, exist_ok=True)

In [ ]:
%pip install -q scikit-learn mpmath

In [ ]:
cmd = f'''
set -o pipefail
PYTHONUNBUFFERED=1 python experiments/measure_plane_fit_crossover.py --out {OUT} 2>&1 | tee {OUT}/crossover.log
'''
rc = subprocess.run(["bash", "-lc", cmd], cwd=REPO).returncode
assert rc == 0, "merenje nije prošlo (provera slaganja A2 i G ili greška); pošalji crossover.log"

In [ ]:
import json
import pandas as pd
from IPython.display import display

c = pd.read_csv(f"{OUT}/crossover.csv")
for dt in ("float64", "float32"):
    print(dt, "speedup of eager A2 over G (median), per round")
    display(c[c.dtype == dt].pivot(index="round", columns="n", values="speedup_A2_vs_G").round(2))
print(json.dumps(json.load(open(f"{OUT}/thresholds.json"))["thresholds"], indent=2))

In [ ]:
from zipfile import ZIP_DEFLATED, ZipFile
from google.colab import files

archive = "/content/plane_fit_crossover.zip"
with ZipFile(archive, "w", ZIP_DEFLATED) as zf:
    for name in sorted(os.listdir(OUT)):
        zf.write(os.path.join(OUT, name), os.path.join("plane_fit_crossover", name))
files.download(archive)